In [231]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, GroupKFold
from missforest import MissForest
import warnings

In [232]:
PATH_TO_FINAL_DATASET = "data_cleaned/final_dataset.parquet"

In [233]:
df = pd.read_parquet(PATH_TO_FINAL_DATASET, engine='pyarrow')

In [221]:
phq8_demographics = df.drop(columns=[col for col in df.columns if col.startswith('sleep_')])
phq8_demographics = phq8_demographics.drop(columns = ["timeInterval_ID"])

In [222]:
X = phq8_demographics.drop(columns=['phq8_score'])
y = phq8_demographics['phq8_score']

In [155]:
demographic_cols = [
    'age',
    'gender_at_birth',
    'marital_status',
    'children',
    'ethnicity_1',
    'eduction_finish_age',
    'drop_school_1',
    'income_gbp',
    'accommodation',
    'mh_smoking',
    'audit_1',
    'depression_medication'
]

In [156]:
categorical_cols = ['marital_status', 'ethnicity_1', 'accommodation']


In [157]:
X['marital_status'].cat.codes.unique()

array([ 2,  5,  0,  1,  4,  3, -1], dtype=int8)

In [85]:
for col in categorical_cols:
    X[col] = X[col].cat.codes
    X[col] = X[col].replace(-1, np.nan)

In [86]:
ordinal_cols = ['income_gbp', 'mh_smoking', 'audit_1']

In [87]:
for col in ordinal_cols:
    X[col] = X[col].cat.codes
    X[col] = X[col].replace(-1, np.nan)

In [224]:
X['gender_at_birth'].cat.codes.unique()

array([ 1, -1,  0], dtype=int8)

In [225]:
X['children'].cat.codes.unique()

array([ 1,  0, -1], dtype=int8)

In [227]:
X['drop_school_1'].cat.codes.unique()

array([ 0, -1,  1], dtype=int8)

In [228]:
X['depression_medication'].cat.codes.unique()

array([-1,  0,  1], dtype=int8)

In [88]:
X['gender_at_birth'] = X['gender_at_birth'].map({'Female': 0, 'Male': 1})
X['children'] = X['children'].map({'No': 0, 'Yes': 1})
X['drop_school_1'] = X['drop_school_1'].map({'No': 0, 'Yes': 1})
X['depression_medication'] = X['depression_medication'].map({'No': 0, 'Yes': 1})

In [89]:
X['gender_at_birth'] = X['gender_at_birth'].astype(np.float64)  
X['depression_medication'] = X['depression_medication'].astype(np.float64)
X['children'] = X['children'].astype(np.float64)
X['drop_school_1'] = X['drop_school_1'].astype(np.float64)

In [90]:
demographics = X.groupby('participant_name')[demographic_cols].first()

In [91]:
imputer = MissForest(categorical = categorical_cols)

In [92]:
imputer.fit(demographics)

/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/missforest/missforest.py:333: UserWarning: Label encoding is no longer performed by default. Users will have to perform categorical features encoding by themselves.
  warnings.warn("Label encoding is no longer performed by default. "
100%|██████████| 5/5 [00:40<00:00,  8.13s/it]


In [93]:
demographics_imputed = imputer.transform(demographics)

/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/missforest/missforest.py:490: UserWarning: Label encoding is no longer performed by default. Users will have to perform categorical features encoding by themselves.
  warnings.warn("Label encoding is no longer performed by default. "
/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/missforest/missforest.py:494: UserWarning: In version 4.2.3, estimator fitting process is moved to `fit` method. `MissForest` will now imputes unseen missing values with fitted estimators with `transform` method. To retain the old behaviour, use `fit_transform` to fit the whole unseen data instead.
  warnings.warn(f"In version {VERSION}, estimator fitting process "
100%|██████████| 5/5 [00:00<00:00, 42.68it/s]


In [94]:
type(demographics_imputed)

pandas.DataFrame

In [95]:
demographics_imputed.isnull().sum()

age                      0
marital_status           0
accommodation            0
mh_smoking               0
children                 0
eduction_finish_age      0
audit_1                  0
depression_medication    0
gender_at_birth          0
ethnicity_1              0
drop_school_1            0
income_gbp               0
dtype: int64

In [96]:
for col in demographics_imputed.columns:
    print(col)
    print(demographics_imputed[col].unique())
    print("\n")

age
[55.         56.         34.         62.         20.         26.
 58.         37.         38.         41.         52.         72.
 22.         23.         61.         32.         47.         63.
 46.         66.         51.         59.         40.         28.
 24.         53.         44.         21.         57.         54.
 29.         50.         64.         42.         35.         31.
 45.         30.         48.         27.         25.         69.
 19.         39.         65.         70.         67.         36.
 60.         49.         68.         43.         33.         71.
 18.         76.         50.29600036 75.        ]


marital_status
[2. 1. 5. 0. 4. 3.]


accommodation
[4. 2. 1. 0. 5. 3. 6. 7.]


mh_smoking
[2.         1.         0.         1.41059073]


children
[1.         0.         0.45595512 0.02910472]


eduction_finish_age
[27.         21.         19.         20.         24.         29.
 16.         18.         25.         22.         23.         17.
 28.         2

In [97]:
X_no_demo = X.drop(columns=demographic_cols)


In [98]:
demographics_imputed.head(5)

,age,marital_status,accommodation,mh_smoking,children,eduction_finish_age,audit_1,depression_medication,gender_at_birth,ethnicity_1,drop_school_1,income_gbp
participant_name,,,,,,,,,,,,
007751c5-d7ad-4bec-a58f-abf32500e2ae,55.0,2.0,4.0,2.0,1.0,27.0,0.0,1.000000,0.159003,0.0,0.046790,2.183756
00bf567f-30ef-4317-9c3d-f62a37bcce27,56.0,1.0,2.0,1.0,1.0,21.0,1.0,1.000000,0.000000,17.0,0.000000,2.115939
0104dfff-4dcd-48ff-b912-51362f098ed0,34.0,1.0,1.0,1.0,1.0,21.0,1.0,0.836573,0.000000,17.0,0.000000,2.180683
01318193-bb33-4c1b-b1bf-d316a014b5ad,62.0,5.0,0.0,2.0,1.0,19.0,3.0,1.000000,-0.256005,0.0,0.035308,2.038822
0157930f-10f7-4d5f-9575-a65efbd1d505,20.0,0.0,5.0,1.0,0.0,20.0,1.0,1.074364,0.000000,0.0,0.000000,0.000000


In [99]:
X_no_demo.merge(demographics_imputed, left_on='participant_name', right_index=True, how = 'left')

,participant_name,age,marital_status,accommodation,mh_smoking,children,eduction_finish_age,audit_1,depression_medication,gender_at_birth,ethnicity_1,drop_school_1,income_gbp
0,9078ae18-c644-4cdb-a582-6a2aa671ecb4,55.0,2.0,0.0,0.0,1.0,17.0,4.000000,1.065801,0.000000,0.0,0.000000,4.000000
1,9078ae18-c644-4cdb-a582-6a2aa671ecb4,55.0,2.0,0.0,0.0,1.0,17.0,4.000000,1.065801,0.000000,0.0,0.000000,4.000000
2,9078ae18-c644-4cdb-a582-6a2aa671ecb4,55.0,2.0,0.0,0.0,1.0,17.0,4.000000,1.065801,0.000000,0.0,0.000000,4.000000
3,9078ae18-c644-4cdb-a582-6a2aa671ecb4,55.0,2.0,0.0,0.0,1.0,17.0,4.000000,1.065801,0.000000,0.0,0.000000,4.000000
4,9078ae18-c644-4cdb-a582-6a2aa671ecb4,55.0,2.0,0.0,0.0,1.0,17.0,4.000000,1.065801,0.000000,0.0,0.000000,4.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8354,5f3217f1-c751-4fee-a8c4-21acc5153d58,64.0,0.0,1.0,0.0,1.0,23.0,1.898202,1.000000,0.000000,0.0,0.000000,3.000000
8355,5f3217f1-c751-4fee-a8c4-21acc5153d58,64.0,0.0,1.0,0.0,1.0,23.0,1.898202,1.000000,0.000000,0.0,0.000000,3.000000
8356,5f3217f1-c751-4fee-a8c4-21acc5153d58,64.0,0.0,1.0,0.0,1.0,23.0,1.898202,1.000000,0.000000,0.0,0.000000,3.000000
8357,6b52c798-bf6e-4512-af0d-98e39939fd37,40.0,4.0,4.0,2.0,1.0,22.0,0.000000,1.000000,-0.135824,0.0,0.015782,1.729068


In [100]:
class ImputerMerger(BaseEstimator, TransformerMixin):
    """
    Single transformer that:
    1. Extracts unique demographic rows (one per participant)
    2. Imputes using MissForest (handles mixed data types automatically)
    3. Merges with physiological data
    """
    
    def __init__(self, demographic_cols, categorical_cols,
                 participant_id_col='participant_name'):
        self.demographic_cols = demographic_cols
        self.categorical_cols = categorical_cols 
        self.participant_id_col = participant_id_col
        
        # Will be fitted
        self.imputer_ = None
        
    def fit(self, X):
        """Fit MissForest on unique demographic rows"""
        # Extract unique demographic rows (one per participant)
        demo_unique = X.groupby(self.participant_id_col)[self.demographic_cols].first()
        
        # Create and fit MissForest imputer
        self.imputer_ = MissForest (
            categorical=self.categorical_cols
        )

        # Fit on demographic data
        self.imputer_.fit(demo_unique)
        
        return self
    
    def transform(self, X):
        """Impute demographics and merge back"""
        
        # Extract unique demographic rows with participant IDs
        demo_unique = X.groupby(self.participant_id_col)[self.demographic_cols].first()
        
        # Impute
        demo_imputed = self.imputer_.transform(demo_unique)
        
        # Drop original demographic columns from X
        X_no_demo = X.drop(columns=self.demographic_cols)
        
        # Merge imputed demographics back
        X_merged = X_no_demo.merge(
            demo_imputed,
            left_on=self.participant_id_col,
            right_index=True,
            how='left'
        )
        
        
        return X_merged

In [101]:
imputer_merger = MissForestImputerMerger(
    demographic_cols=demographic_cols,
    categorical_cols=categorical_cols,
)

In [102]:
imputer_merger.fit(X)

/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/missforest/missforest.py:333: UserWarning: Label encoding is no longer performed by default. Users will have to perform categorical features encoding by themselves.
  warnings.warn("Label encoding is no longer performed by default. "
100%|██████████| 5/5 [01:03<00:00, 12.64s/it]


,demographic_cols,"['age', 'gender_at_birth', ...]"
,categorical_cols,"['marital_status', 'ethnicity_1', ...]"
,participant_id_col,'participant_name'


In [103]:
X_imputed = imputer_merger.transform(X)

/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/missforest/missforest.py:490: UserWarning: Label encoding is no longer performed by default. Users will have to perform categorical features encoding by themselves.
  warnings.warn("Label encoding is no longer performed by default. "
/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/missforest/missforest.py:494: UserWarning: In version 4.2.3, estimator fitting process is moved to `fit` method. `MissForest` will now imputes unseen missing values with fitted estimators with `transform` method. To retain the old behaviour, use `fit_transform` to fit the whole unseen data instead.
  warnings.warn(f"In version {VERSION}, estimator fitting process "
100%|██████████| 5/5 [00:00<00:00, 46.29it/s]


In [104]:
imputed_demos = X_imputed.groupby('participant_name')[demographic_cols].first()

In [105]:
imputed_demos.isnull().sum()

age                      0
gender_at_birth          0
marital_status           0
children                 0
ethnicity_1              0
eduction_finish_age      0
drop_school_1            0
income_gbp               0
accommodation            0
mh_smoking               0
audit_1                  0
depression_medication    0
dtype: int64

In [106]:
for col in imputed_demos.columns:
    print(col)
    print(imputed_demos[col].unique())
    print("\n")

age
[55.         56.         34.         62.         20.         26.
 58.         37.         38.         41.         52.         72.
 22.         23.         61.         32.         47.         63.
 46.         66.         51.         59.         40.         28.
 24.         53.         44.         21.         57.         54.
 29.         50.         64.         42.         35.         31.
 45.         30.         48.         27.         25.         69.
 19.         39.         65.         70.         67.         36.
 60.         49.         68.         43.         33.         71.
 18.         76.         50.29600036 75.        ]


gender_at_birth
[ 1.59003043e-01  0.00000000e+00 -2.56004817e-01  1.00000000e+00
  2.04654993e-01 -8.70531555e-02  7.93779455e-02  8.76757988e-03
  3.30895789e-01  1.48071683e-01  1.10506914e-01  1.97949486e-01
  3.59546398e-02 -9.00747431e-02 -5.66295110e-02 -8.90523063e-02
 -4.95088873e-02  8.82787068e-02  3.98787270e-01  8.44193143e-02
  7.22114051e-03 -

In [107]:
X_imputed.isnull().sum()

participant_name         0
age                      0
marital_status           0
accommodation            0
mh_smoking               0
children                 0
eduction_finish_age      0
audit_1                  0
depression_medication    0
gender_at_birth          0
ethnicity_1              0
drop_school_1            0
income_gbp               0
dtype: int64

In [ ]:
categorical_cols = ['marital_status', 'ethnicity_1', 'accommodation', ]


In [125]:
boolean_cols = ['children', 'drop_school_1', 'depression_medication', 'gender_at_birth']

In [110]:
from scripts.classes.ImputerMerger import ImputerMerger

In [126]:
imputer_merger = ImputerMerger(
    demographic_cols=demographic_cols,
    categorical_cols=categorical_cols + boolean_cols
)

In [127]:
imputer_merger.fit(X)

/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/missforest/missforest.py:333: UserWarning: Label encoding is no longer performed by default. Users will have to perform categorical features encoding by themselves.
  warnings.warn("Label encoding is no longer performed by default. "
  0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:44<00:00,  8.88s/it]


,demographic_cols,"['age', 'gender_at_birth', ...]"
,categorical_cols,"['marital_status', 'ethnicity_1', ...]"
,participant_id_col,'participant_name'


In [128]:
X_imputed = imputer_merger.transform(X)

/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/missforest/missforest.py:490: UserWarning: Label encoding is no longer performed by default. Users will have to perform categorical features encoding by themselves.
  warnings.warn("Label encoding is no longer performed by default. "
/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/missforest/missforest.py:494: UserWarning: In version 4.2.3, estimator fitting process is moved to `fit` method. `MissForest` will now imputes unseen missing values with fitted estimators with `transform` method. To retain the old behaviour, use `fit_transform` to fit the whole unseen data instead.
  warnings.warn(f"In version {VERSION}, estimator fitting process "
100%|██████████| 5/5 [00:00<00:00, 48.15it/s]


In [118]:
X["participant_name"].unique()

<ArrowStringArray>
['9078ae18-c644-4cdb-a582-6a2aa671ecb4',
 '8fc6f0c6-4b6c-45a4-8140-ee53a23caa89',
 '90230df5-657d-4820-b624-e192e928ed6c',
 '929e0e36-8b80-4749-8c5e-3c540061d52a',
 '8fc54c80-6b39-449e-bb45-dea9e7ace262',
 '8dc143ea-c1e5-42ac-a74b-3dd3c15ceabb',
 '8de38717-8b8b-498b-a986-6da43e807529',
 '8e0e62d2-cad4-454d-88e2-2824323012d5',
 '8fa960a7-a9f7-488f-a5a2-57c2400b6ba6',
 '94806f1d-836f-4653-ac0d-4de989bb945e',
 ...
 '49a29158-e258-4949-bb1e-ee4292ed5243',
 '05f44e49-1b4e-4336-b962-d3eff448ab93',
 '02919d70-343c-4efa-97c0-c34bf85546f0',
 '5fc26e60-e464-4dd4-b3ca-ec539bc827cc',
 'ab6805e7-18e8-45c8-8f10-b75b3d034fc0',
 'b12be729-f2bf-47bb-a7f9-7a684ea8cc9c',
 '3da1638f-9904-408a-a621-8141e76e7e0f',
 '538dbda6-fb74-4e46-856a-de39ce6064f9',
 '748b4be9-05bc-4ead-93f9-a2926448c100',
 '811257ba-fd44-4b22-b241-b83f77b2f9ae']
Length: 509, dtype: str

In [129]:
X[X['participant_name'] == '8fc54c80-6b39-449e-bb45-dea9e7ace262']

,participant_name,age,gender_at_birth,marital_status,children,ethnicity_1,eduction_finish_age,drop_school_1,income_gbp,accommodation,mh_smoking,audit_1,depression_medication
64,8fc54c80-6b39-449e-bb45-dea9e7ace262,65.0,NaN,0.0,1.0,NaN,19.0,NaN,NaN,4.0,2.0,2.0,1.0
78,8fc54c80-6b39-449e-bb45-dea9e7ace262,65.0,NaN,0.0,1.0,NaN,19.0,NaN,NaN,4.0,2.0,2.0,1.0
91,8fc54c80-6b39-449e-bb45-dea9e7ace262,65.0,NaN,0.0,1.0,NaN,19.0,NaN,NaN,4.0,2.0,2.0,1.0
92,8fc54c80-6b39-449e-bb45-dea9e7ace262,65.0,NaN,0.0,1.0,NaN,19.0,NaN,NaN,4.0,2.0,2.0,1.0
93,8fc54c80-6b39-449e-bb45-dea9e7ace262,65.0,NaN,0.0,1.0,NaN,19.0,NaN,NaN,4.0,2.0,2.0,1.0
94,8fc54c80-6b39-449e-bb45-dea9e7ace262,65.0,NaN,0.0,1.0,NaN,19.0,NaN,NaN,4.0,2.0,2.0,1.0
95,8fc54c80-6b39-449e-bb45-dea9e7ace262,65.0,NaN,0.0,1.0,NaN,19.0,NaN,NaN,4.0,2.0,2.0,1.0
7550,8fc54c80-6b39-449e-bb45-dea9e7ace262,65.0,NaN,0.0,1.0,NaN,19.0,NaN,NaN,4.0,2.0,2.0,1.0
7551,8fc54c80-6b39-449e-bb45-dea9e7ace262,65.0,NaN,0.0,1.0,NaN,19.0,NaN,NaN,4.0,2.0,2.0,1.0
7554,8fc54c80-6b39-449e-bb45-dea9e7ace262,65.0,NaN,0.0,1.0,NaN,19.0,NaN,NaN,4.0,2.0,2.0,1.0


In [130]:
X_imputed[X_imputed['participant_name'] == '8fc54c80-6b39-449e-bb45-dea9e7ace262']

,participant_name,age,marital_status,accommodation,mh_smoking,children,eduction_finish_age,audit_1,depression_medication,gender_at_birth,ethnicity_1,drop_school_1,income_gbp
64,8fc54c80-6b39-449e-bb45-dea9e7ace262,65.0,0.0,4.0,2.0,1.0,19.0,2.0,1.0,0.0,0.0,0.0,1.783988
78,8fc54c80-6b39-449e-bb45-dea9e7ace262,65.0,0.0,4.0,2.0,1.0,19.0,2.0,1.0,0.0,0.0,0.0,1.783988
91,8fc54c80-6b39-449e-bb45-dea9e7ace262,65.0,0.0,4.0,2.0,1.0,19.0,2.0,1.0,0.0,0.0,0.0,1.783988
92,8fc54c80-6b39-449e-bb45-dea9e7ace262,65.0,0.0,4.0,2.0,1.0,19.0,2.0,1.0,0.0,0.0,0.0,1.783988
93,8fc54c80-6b39-449e-bb45-dea9e7ace262,65.0,0.0,4.0,2.0,1.0,19.0,2.0,1.0,0.0,0.0,0.0,1.783988
94,8fc54c80-6b39-449e-bb45-dea9e7ace262,65.0,0.0,4.0,2.0,1.0,19.0,2.0,1.0,0.0,0.0,0.0,1.783988
95,8fc54c80-6b39-449e-bb45-dea9e7ace262,65.0,0.0,4.0,2.0,1.0,19.0,2.0,1.0,0.0,0.0,0.0,1.783988
7550,8fc54c80-6b39-449e-bb45-dea9e7ace262,65.0,0.0,4.0,2.0,1.0,19.0,2.0,1.0,0.0,0.0,0.0,1.783988
7551,8fc54c80-6b39-449e-bb45-dea9e7ace262,65.0,0.0,4.0,2.0,1.0,19.0,2.0,1.0,0.0,0.0,0.0,1.783988
7554,8fc54c80-6b39-449e-bb45-dea9e7ace262,65.0,0.0,4.0,2.0,1.0,19.0,2.0,1.0,0.0,0.0,0.0,1.783988


In [131]:
from sklearn.compose import ColumnTransformer

In [132]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [133]:
from sklearn.pipeline import Pipeline

In [ ]:

ct = ColumnTransformer(
    transformers=[
        ('remove_participant_name', 'drop', ['participant_name'])
    ],
    remainder='passthrough',
    verbose_feature_names_out=False 
).set_output(transform="pandas")

In [142]:
X_dropped = ct.fit_transform(X_imputed)

In [143]:
X_dropped

,age,marital_status,accommodation,mh_smoking,children,eduction_finish_age,audit_1,depression_medication,gender_at_birth,ethnicity_1,drop_school_1,income_gbp
0,55.0,2.0,0.0,0.0,1.0,17.0,4.000000,1.0,0.0,0.0,0.0,4.000000
1,55.0,2.0,0.0,0.0,1.0,17.0,4.000000,1.0,0.0,0.0,0.0,4.000000
2,55.0,2.0,0.0,0.0,1.0,17.0,4.000000,1.0,0.0,0.0,0.0,4.000000
3,55.0,2.0,0.0,0.0,1.0,17.0,4.000000,1.0,0.0,0.0,0.0,4.000000
4,55.0,2.0,0.0,0.0,1.0,17.0,4.000000,1.0,0.0,0.0,0.0,4.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
8354,64.0,0.0,1.0,0.0,1.0,23.0,2.262071,1.0,0.0,0.0,0.0,3.000000
8355,64.0,0.0,1.0,0.0,1.0,23.0,2.262071,1.0,0.0,0.0,0.0,3.000000
8356,64.0,0.0,1.0,0.0,1.0,23.0,2.262071,1.0,0.0,0.0,0.0,3.000000
8357,40.0,4.0,4.0,2.0,1.0,22.0,0.000000,1.0,0.0,0.0,0.0,1.425809


In [149]:
for col in categorical_cols:
    print(col)
    print(X_imputed[col].unique())
    print("\n")

marital_status
[2. 5. 0. 1. 4. 3.]


ethnicity_1
[ 0. 17. 16.  8.  2. 19.  3.  6. 15.  4.  5.  7. 13. 18. 12. 10. 11. 14.
  1.  9.]


accommodation
[0. 3. 6. 2. 4. 1. 5. 7.]




In [162]:
X['marital_status'].cat.codes.unique().tolist()

[2, 5, 0, 1, 4, 3, -1]

In [161]:
marital_status_numeric_categories = [
    x for x in X['marital_status'].cat.codes.unique().tolist() if x != -1
]

In [163]:
marital_status_numeric_categories

[2, 5, 0, 1, 4, 3]

In [178]:
categorical_cols_dict = {
    col: sorted([x for x in X[col].cat.codes.unique().tolist() if x != -1])
    for col in categorical_cols
}

In [179]:
categories_list = list(categorical_cols_dict.values())

In [180]:
categories_list

[[0, 1, 2, 3, 4, 5],
 [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19],
 [0, 1, 2, 3, 4, 5, 6, 7]]

In [ ]:
oh_encoder = OneHotEncoder(
    categories=categories_list,
    sparse_output=False,
    drop='first'                    # drop first to avoid dummy variable trap
).set_output(transform="pandas")

In [194]:
# use one hot encoder to encode the categorical columns in X_dropped
X_encoded = oh_encoder.fit_transform(X_dropped[categorical_cols])

In [195]:
X_encoded.columns

Index(['marital_status_1.0', 'marital_status_2.0', 'marital_status_3.0',
       'marital_status_4.0', 'marital_status_5.0', 'ethnicity_1_1.0',
       'ethnicity_1_2.0', 'ethnicity_1_3.0', 'ethnicity_1_4.0',
       'ethnicity_1_5.0', 'ethnicity_1_6.0', 'ethnicity_1_7.0',
       'ethnicity_1_8.0', 'ethnicity_1_9.0', 'ethnicity_1_10.0',
       'ethnicity_1_11.0', 'ethnicity_1_12.0', 'ethnicity_1_13.0',
       'ethnicity_1_14.0', 'ethnicity_1_15.0', 'ethnicity_1_16.0',
       'ethnicity_1_17.0', 'ethnicity_1_18.0', 'ethnicity_1_19.0',
       'accommodation_1.0', 'accommodation_2.0', 'accommodation_3.0',
       'accommodation_4.0', 'accommodation_5.0', 'accommodation_6.0',
       'accommodation_7.0'],
      dtype='str')

In [196]:
numerical_cols = ['eduction_finish_age', 'age']


In [197]:
boolean_cols

['children', 'drop_school_1', 'depression_medication', 'gender_at_birth']

In [198]:
ordinal_cols

['income_gbp', 'mh_smoking', 'audit_1']

In [212]:
column_processor = ColumnTransformer(
    transformers=[
        # One-Hot Encode Categorical and Boolean columns
        ('cat', oh_encoder, categorical_cols),
        
        # Scale Numerical and Ordinal columns
        ('num', StandardScaler(), numerical_cols + ordinal_cols),
        
        # Explicitly drop the participant_name now that merging is done
        ('drop_id', 'drop', ['participant_name'])
    ],
    remainder='passthrough',
    verbose_feature_names_out=False 
).set_output(transform="pandas")

In [213]:
X_preprocessed = column_processor.fit_transform(X_imputed)

In [214]:
X_preprocessed.head()

,marital_status_1.0,marital_status_2.0,marital_status_3.0,marital_status_4.0,marital_status_5.0,ethnicity_1_1.0,ethnicity_1_2.0,ethnicity_1_3.0,ethnicity_1_4.0,ethnicity_1_5.0,...,accommodation_7.0,eduction_finish_age,age,income_gbp,mh_smoking,audit_1,children,depression_medication,gender_at_birth,drop_school_1
0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,-0.615066,0.480064,1.847368,-0.824837,1.771746,1.0,1.0,0.0,0.0
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,-0.615066,0.480064,1.847368,-0.824837,1.771746,1.0,1.0,0.0,0.0
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,-0.615066,0.480064,1.847368,-0.824837,1.771746,1.0,1.0,0.0,0.0
3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,-0.615066,0.480064,1.847368,-0.824837,1.771746,1.0,1.0,0.0,0.0
4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,-0.615066,0.480064,1.847368,-0.824837,1.771746,1.0,1.0,0.0,0.0


In [209]:
X_preprocessed.columns

Index(['marital_status_1.0', 'marital_status_2.0', 'marital_status_3.0',
       'marital_status_4.0', 'marital_status_5.0', 'ethnicity_1_1.0',
       'ethnicity_1_2.0', 'ethnicity_1_3.0', 'ethnicity_1_4.0',
       'ethnicity_1_5.0', 'ethnicity_1_6.0', 'ethnicity_1_7.0',
       'ethnicity_1_8.0', 'ethnicity_1_9.0', 'ethnicity_1_10.0',
       'ethnicity_1_11.0', 'ethnicity_1_12.0', 'ethnicity_1_13.0',
       'ethnicity_1_14.0', 'ethnicity_1_15.0', 'ethnicity_1_16.0',
       'ethnicity_1_17.0', 'ethnicity_1_18.0', 'ethnicity_1_19.0',
       'accommodation_1.0', 'accommodation_2.0', 'accommodation_3.0',
       'accommodation_4.0', 'accommodation_5.0', 'accommodation_6.0',
       'accommodation_7.0', 'eduction_finish_age', 'age', 'income_gbp',
       'mh_smoking', 'audit_1', 'children', 'depression_medication',
       'gender_at_birth', 'drop_school_1'],
      dtype='str')

In [211]:
# subset columns that start with marital
marital_cols = [col for col in X_preprocessed.columns if col.startswith('marital_status')]
X_preprocessed[marital_cols].head()

,marital_status_1.0,marital_status_2.0,marital_status_3.0,marital_status_4.0,marital_status_5.0
0,0.0,1.0,0.0,0.0,0.0
1,0.0,1.0,0.0,0.0,0.0
2,0.0,1.0,0.0,0.0,0.0
3,0.0,1.0,0.0,0.0,0.0
4,0.0,1.0,0.0,0.0,0.0


In [215]:
pipeline = Pipeline(steps=[
    ('imputer_merger', ImputerMerger(
        demographic_cols=demographic_cols,
        categorical_cols=categorical_cols + boolean_cols
    )),
    ('preprocessing', column_processor)
])

,participant_name,age,gender_at_birth,marital_status,children,ethnicity_1,eduction_finish_age,drop_school_1,income_gbp,accommodation,mh_smoking,audit_1,depression_medication
0,9078ae18-c644-4cdb-a582-6a2aa671ecb4,55.0,Female,Married and living with your husband/wife,Yes,English/Welsh/Scottish/Northern Irish/British,17.0,No,"More than £55,000",Own outright,"No, I have never smoked",4 or more times a week,NaN
1,9078ae18-c644-4cdb-a582-6a2aa671ecb4,55.0,Female,Married and living with your husband/wife,Yes,English/Welsh/Scottish/Northern Irish/British,17.0,No,"More than £55,000",Own outright,"No, I have never smoked",4 or more times a week,NaN
2,9078ae18-c644-4cdb-a582-6a2aa671ecb4,55.0,Female,Married and living with your husband/wife,Yes,English/Welsh/Scottish/Northern Irish/British,17.0,No,"More than £55,000",Own outright,"No, I have never smoked",4 or more times a week,NaN
3,9078ae18-c644-4cdb-a582-6a2aa671ecb4,55.0,Female,Married and living with your husband/wife,Yes,English/Welsh/Scottish/Northern Irish/British,17.0,No,"More than £55,000",Own outright,"No, I have never smoked",4 or more times a week,NaN
4,9078ae18-c644-4cdb-a582-6a2aa671ecb4,55.0,Female,Married and living with your husband/wife,Yes,English/Welsh/Scottish/Northern Irish/British,17.0,No,"More than £55,000",Own outright,"No, I have never smoked",4 or more times a week,NaN


In [216]:
X_imputed_preprocessed = pipeline.fit_transform(X)

/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/missforest/missforest.py:333: UserWarning: Label encoding is no longer performed by default. Users will have to perform categorical features encoding by themselves.
  warnings.warn("Label encoding is no longer performed by default. "


TypeError: Object with dtype category cannot perform the numpy op isinf

In [235]:
def prepare_data(df, categorical_cols, ordinal_cols, boolean_cols):
    phq8_demographics = df.drop(columns=[col for col in df.columns if col.startswith('sleep_')])
    phq8_demographics = phq8_demographics.drop(columns = ["timeInterval_ID"])

    X = phq8_demographics.drop(columns=['phq8_score'])
    y = phq8_demographics['phq8_score']

    cols_to_encode = categorical_cols + ordinal_cols + boolean_cols

    for col in cols_to_encode:
        X[col] = X[col].cat.codes
        X[col] = X[col].replace(-1, np.nan)

    return X, y

In [236]:
X, y = prepare_data(df, categorical_cols, ordinal_cols, boolean_cols)

In [237]:
X.head()

,participant_name,age,gender_at_birth,marital_status,children,ethnicity_1,eduction_finish_age,drop_school_1,income_gbp,accommodation,mh_smoking,audit_1,depression_medication
0,9078ae18-c644-4cdb-a582-6a2aa671ecb4,55.0,1.0,2.0,1.0,0.0,17.0,0.0,4.0,0.0,0.0,4.0,NaN
1,9078ae18-c644-4cdb-a582-6a2aa671ecb4,55.0,1.0,2.0,1.0,0.0,17.0,0.0,4.0,0.0,0.0,4.0,NaN
2,9078ae18-c644-4cdb-a582-6a2aa671ecb4,55.0,1.0,2.0,1.0,0.0,17.0,0.0,4.0,0.0,0.0,4.0,NaN
3,9078ae18-c644-4cdb-a582-6a2aa671ecb4,55.0,1.0,2.0,1.0,0.0,17.0,0.0,4.0,0.0,0.0,4.0,NaN
4,9078ae18-c644-4cdb-a582-6a2aa671ecb4,55.0,1.0,2.0,1.0,0.0,17.0,0.0,4.0,0.0,0.0,4.0,NaN


In [241]:
categorical_cols_dict = {
    col: sorted(X[col].dropna().unique().tolist())
    for col in categorical_cols
}

In [242]:
categorical_cols_dict

{'marital_status': [0.0, 1.0, 2.0, 3.0, 4.0, 5.0],
 'ethnicity_1': [0.0,
  1.0,
  2.0,
  3.0,
  4.0,
  5.0,
  6.0,
  7.0,
  8.0,
  9.0,
  10.0,
  11.0,
  12.0,
  13.0,
  14.0,
  15.0,
  16.0,
  17.0,
  18.0,
  19.0],
 'accommodation': [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]}

In [238]:
X_imputed_preprocessed = pipeline.fit_transform(X)

/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/missforest/missforest.py:333: UserWarning: Label encoding is no longer performed by default. Users will have to perform categorical features encoding by themselves.
  warnings.warn("Label encoding is no longer performed by default. "
100%|██████████| 5/5 [00:42<00:00,  8.41s/it]
/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/missforest/missforest.py:490: UserWarning: Label encoding is no longer performed by default. Users will have to perform categorical features encoding by themselves.
  warnings.warn("Label encoding is no longer performed by default. "
/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/missforest/missforest.py:494: UserWarning: In version 4.2.3, estimator fitting process is moved to `fit` method. `MissForest` will now imputes unseen missing values with fitted estimators with `transform` method. To retain the old behaviour, use `fit_t

In [239]:
X_imputed_preprocessed.head()

,marital_status_1.0,marital_status_2.0,marital_status_3.0,marital_status_4.0,marital_status_5.0,ethnicity_1_1.0,ethnicity_1_2.0,ethnicity_1_3.0,ethnicity_1_4.0,ethnicity_1_5.0,...,accommodation_7.0,eduction_finish_age,age,income_gbp,mh_smoking,audit_1,children,depression_medication,gender_at_birth,drop_school_1
0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,-0.615066,0.480064,1.847368,-0.824837,1.771746,1.0,1.0,1.0,0.0
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,-0.615066,0.480064,1.847368,-0.824837,1.771746,1.0,1.0,1.0,0.0
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,-0.615066,0.480064,1.847368,-0.824837,1.771746,1.0,1.0,1.0,0.0
3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,-0.615066,0.480064,1.847368,-0.824837,1.771746,1.0,1.0,1.0,0.0
4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,-0.615066,0.480064,1.847368,-0.824837,1.771746,1.0,1.0,1.0,0.0


In [240]:
X_imputed_preprocessed.columns

Index(['marital_status_1.0', 'marital_status_2.0', 'marital_status_3.0',
       'marital_status_4.0', 'marital_status_5.0', 'ethnicity_1_1.0',
       'ethnicity_1_2.0', 'ethnicity_1_3.0', 'ethnicity_1_4.0',
       'ethnicity_1_5.0', 'ethnicity_1_6.0', 'ethnicity_1_7.0',
       'ethnicity_1_8.0', 'ethnicity_1_9.0', 'ethnicity_1_10.0',
       'ethnicity_1_11.0', 'ethnicity_1_12.0', 'ethnicity_1_13.0',
       'ethnicity_1_14.0', 'ethnicity_1_15.0', 'ethnicity_1_16.0',
       'ethnicity_1_17.0', 'ethnicity_1_18.0', 'ethnicity_1_19.0',
       'accommodation_1.0', 'accommodation_2.0', 'accommodation_3.0',
       'accommodation_4.0', 'accommodation_5.0', 'accommodation_6.0',
       'accommodation_7.0', 'eduction_finish_age', 'age', 'income_gbp',
       'mh_smoking', 'audit_1', 'children', 'depression_medication',
       'gender_at_birth', 'drop_school_1'],
      dtype='str')